In [ ]:
%pip install transformers

ERROR: Operation cancelled by user


In [ ]:
import pandas as pd
import pickle                # for saving the trained models
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from nltk.tokenize import word_tokenize
from transformers import (DistilBertTokenizer,
                          DistilBertForSequenceClassification)
import warnings
warnings.filterwarnings('ignore')

In [11]:
df = pd.read_csv('preprocessed_resume_data.csv')
df.head()

,cleaned_text,job_position
0,big data analytics working database warehouse ...,Software Development
1,fresher looking join data analyst junior data ...,Data Science
2,software development machine learning deep lea...,Marketing & Sales
3,obtain position fast paced business office env...,Marketing & Sales
4,professional accountant outstanding work ethic...,Mobile Development


In [ ]:
# Encode Target Labels
label_encoder = LabelEncoder()
df["encoded_job_position"] = label_encoder.fit_transform(df["job_position"])

# save the label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Label encoder saved successfully!")

In [13]:
# features and labels
X = df["cleaned_text"]              # Features (text data)
y = df["encoded_job_position"]       # Target (encoded job positions)
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )

In [14]:
# ----------
# LSTM Model
# ----------

# Split text into words
train_tokens = X_train.apply( lambda x: x.split() )
test_tokens = X_test.apply( lambda x: x.split() )

In [15]:
# Build Vocabulary
vocab = {}
for tokens in train_tokens:
    for word in tokens:
        if word not in vocab:
            vocab[word] = len(vocab) + 1  # Start indexing from 1

print("Vocabulary Size:")
print(len(vocab))

Vocabulary Size:
2786


In [16]:
# convert tokens into integer sequences
def text_to_sequence(tokens):
    return [vocab.get(word, 0) for word in tokens]  # Use 0 for unknown words
train_sequences = train_tokens.apply(text_to_sequence)
test_sequences = test_tokens.apply(text_to_sequence)

In [17]:
# create padded sequences
max_length= 100  # maximum sequence length
def pad_sequence(sequence):
    if len(sequence) < max_length:
        return sequence + [0] * (max_length - len(sequence))  # Pad with 0s
    else:
        return sequence[:max_length]  # Truncate if longer than max_length
padded_train_sequences = train_sequences.apply(pad_sequence)
padded_test_sequences = test_sequences.apply(pad_sequence)

In [18]:
# Combine padded sequences to match the original dataframe length
all_padded_sequences = pd.concat([padded_train_sequences, padded_test_sequences]).sort_index()

# Convert to PyTorch Tensors using the full aligned dataset
X = torch.tensor(all_padded_sequences.tolist(), dtype=torch.long)
y = torch.tensor(df["encoded_job_position"].values, dtype=torch.long)

In [19]:
# Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
# Create DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [21]:
# initialize LSTM model
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        hidden = hidden.squeeze(0)
        predictions = self.fc(hidden)
        return predictions

In [22]:
# Initialize Model, Loss Function, and Optimizer
vocab_size = len(vocab) + 1  # +1 for padding token
embedding_dim = 128
hidden_dim = 128
output_dim = len(label_encoder.classes_)  # Number of job categories

model = LSTMClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
# Training Loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10, Loss: 1.8394
Epoch 2/10, Loss: 0.2170
Epoch 3/10, Loss: 0.1468
Epoch 4/10, Loss: 0.1356
Epoch 5/10, Loss: 0.1297
Epoch 6/10, Loss: 0.1337
Epoch 7/10, Loss: 0.1267
Epoch 8/10, Loss: 0.1250
Epoch 9/10, Loss: 0.1247
Epoch 10/10, Loss: 0.1233


In [26]:
# evaluation mode
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        _, predicted_labels = torch.max(predictions, 1)
        total += y_batch.size(0)
        correct += (predicted_labels == y_batch).sum().item()

In [27]:
# accuracy calculation
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9388


In [28]:
# ----------
# BERT Model
# ----------

# initialize BERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [29]:
# create BERT dataset
class BERTDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Use the tokenizer call method directly
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [30]:
# DistilBERT model initialization
bert_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=output_dim)

# fine-tuning BERT model
optimizer = AdamW(bert_model.parameters(), lr=2e-5)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
# 1. Prepare BERT DataLoaders
train_texts, test_texts, train_labels, test_labels = train_test_split(df['cleaned_text'].tolist(), df['encoded_job_position'].tolist(), test_size=0.2, random_state=42)

train_bert_dataset = BERTDataset(train_texts, train_labels, tokenizer)
test_bert_dataset = BERTDataset(test_texts, test_labels, tokenizer)

bert_train_loader = DataLoader(train_bert_dataset, batch_size=16, shuffle=True)
bert_test_loader = DataLoader(test_bert_dataset, batch_size=16)

# 2. BERT training loop
num_epochs = 5
bert_model.to('cuda' if torch.cuda.is_available() else 'cpu')

for epoch in range(num_epochs):
    bert_model.train()
    total_loss = 0
    for batch in bert_train_loader:
        optimizer.zero_grad()

        # Move batch to device
        input_ids = batch['input_ids'].to(bert_model.device)
        attention_mask = batch['attention_mask'].to(bert_model.device)
        labels = batch['labels'].to(bert_model.device)

        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(bert_train_loader):.4f}")

Epoch 1/5, Loss: 0.4876
Epoch 2/5, Loss: 0.1108
Epoch 3/5, Loss: 0.0956
Epoch 4/5, Loss: 0.0889
Epoch 5/5, Loss: 0.0879


In [32]:
# BERT evaluation
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
bert_model.eval()

correct, total = 0, 0
with torch.no_grad():
    for batch in bert_test_loader:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        _, predicted_labels = torch.max(outputs.logits, 1)

        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()

# accuracy calculation
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9541


In [ ]:
bert_model.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")
print("BERT model saved successfully!")

BERT model saved successfully!
